# Predicting Electric Vehicle Purchases: Baseline Modeling

Kaggle Playground Series S6E9. Implements Phase 2 of
`docs/3_implementation_plan.md`: v1 sanity baselines and v2 strong models on
**fold definition F1** (`docs/4_experiment_ledger.md`), with the first-fit
runtime/memory measurement required by the scale override in
`docs/0_coding_standards.md`. Every run — including rejected ones — gets a
ledger row. The working champion's fold-mean test predictions are written to
`submission.csv`.

EDA context (`docs/2_eda_insights.md`): top-heavy signal
(`Environmental_Concern_Level`, `Subsidy_Available`, `Annual_Income_USD`),
strictly monotone ordinals, one big interaction (the subsidy gate), no
missing values, no drift.

## 1. Config

In [1]:
import json
import platform
import resource
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import lightgbm as lgb
from catboost import CatBoostClassifier

SEED = 42
N_SPLITS = 5  # fold definition F1 -- docs/4_experiment_ledger.md
TARGET = "Will_Buy_EV"
POSITIVE_CLASS = "Yes"
NOTEBOOK_VERSION = "v1"

# Mode flags (master standard §4): flip off to skip expensive sections.
RUN_V1_SANITY = True
RUN_V2_STRONG = True
RUN_ANX_CATEGORICAL_AB = True
RUN_SUBMISSION = True

NUMERIC_FEATURES = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]
BASE_CATEGORICALS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
]
ANX = "Range_Anxiety_Level"
ANX_MAP = {"Low": 0, "Medium": 1, "High": 2}  # monotone -- EDA §4
ALL_FEATURES = NUMERIC_FEATURES + BASE_CATEGORICALS + [ANX]

print("python", platform.python_version())
print({m.__name__: m.__version__ for m in (np, pd, sklearn, lgb)})

python 3.9.6
{'numpy': '2.0.2', 'pandas': '2.3.3', 'sklearn': '1.6.1', 'lightgbm': '4.6.0'}


## 2. Data Loading & Feature Frames

`Range_Anxiety_Level` is ordinal-encoded by default (strictly monotone
target rate — EDA §4); Section 6 A/Bs the categorical treatment. The other
five categoricals stay native (`category` dtype for HGB/LightGBM; named
`cat_features` for CatBoost).

In [2]:
_CANDIDATE_DIRS = [
    Path("/kaggle/input/playground-series-s6e9"),
    Path("../data"),
    Path("data"),
]
DATA_DIR = next(p for p in _CANDIDATE_DIRS if (p / "train.csv").exists())
print(f"DATA_DIR = {DATA_DIR.resolve()}")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert train.shape == (668_665, 15) and test.shape == (286_571, 14)
assert train[ALL_FEATURES].isna().sum().sum() == 0
assert test[ALL_FEATURES].isna().sum().sum() == 0

y = (train[TARGET] == POSITIVE_CLASS).astype(int)


def make_features(
    df: pd.DataFrame, anx_as_categorical: bool = False
) -> pd.DataFrame:
    """Model-ready feature frame.

    Args:
        df: Raw train or test frame.
        anx_as_categorical: If True, keep Range_Anxiety_Level categorical
            instead of the default ordinal int encoding.

    Returns:
        Feature frame with category dtypes on the base categoricals.
    """
    frame = df[ALL_FEATURES].copy()
    if anx_as_categorical:
        frame[ANX] = frame[ANX].astype("category")
    else:
        frame[ANX] = frame[ANX].map(ANX_MAP).astype("int8")
    for col in BASE_CATEGORICALS:
        frame[col] = frame[col].astype("category")
    return frame


X = make_features(train)
X_test = make_features(test)
X_anxcat = make_features(train, anx_as_categorical=True)
X_test_anxcat = make_features(test, anx_as_categorical=True)
print(X.dtypes.to_string())

DATA_DIR = /Users/tuannm3812/Documents/GitHub/2. Kaggle/kaggle-s6e9-predicting-electric-vehicle-purchases/data


Age                               int64
Annual_Income_USD               float64
Daily_Commute_km                float64
Number_of_Cars_Owned              int64
Charging_Stations_Near_Home       int64
Charging_Stations_Near_Work       int64
Environmental_Concern_Level     float64
Gender                         category
City_Type                      category
Current_Car_Type               category
Home_Charging_Possible         category
Subsidy_Available              category
Range_Anxiety_Level                int8


## 3. Cross-Validation Harness (F1)

One harness for every model, so OOF predictions align row-for-row across
candidates (`docs/0_coding_standards.md`). Wall-clock and peak RSS are
recorded per run — the scale-override measurement.

In [3]:
results = []
oof_store = {}
test_store = {}


def peak_rss_gb() -> float:
    """Process peak RSS in GB (ru_maxrss is bytes on macOS, KB on Linux)."""
    raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return raw / (1024**3 if sys.platform == "darwin" else 1024**2)


def run_cv(name, model_factory, X_tr, X_te, cat_features=None):
    """5-fold OOF CV on F1; stores aligned OOF and fold-mean test preds.

    Args:
        name: Run name (becomes the ledger row key).
        model_factory: Zero-arg callable returning a fresh estimator.
        X_tr: Train features aligned with global `y`.
        X_te: Test features.
        cat_features: If set, passed to fit() (CatBoost path).

    Returns:
        (oof, test_pred) arrays.
    """
    skf = StratifiedKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=SEED
    )
    oof = np.zeros(len(X_tr))
    test_pred = np.zeros(len(X_te))
    fold_aucs = []
    t0 = time.time()
    for tr_idx, va_idx in skf.split(X_tr, y):
        model = model_factory()
        if cat_features is None:
            model.fit(X_tr.iloc[tr_idx], y.iloc[tr_idx])
        else:
            model.fit(
                X_tr.iloc[tr_idx], y.iloc[tr_idx],
                cat_features=cat_features,
            )
        oof[va_idx] = model.predict_proba(X_tr.iloc[va_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y.iloc[va_idx], oof[va_idx]))
        test_pred += model.predict_proba(X_te)[:, 1] / N_SPLITS
    elapsed = time.time() - t0
    row = {
        "run": name,
        "oof_auc": float(roc_auc_score(y, oof)),
        "fold_std": float(np.std(fold_aucs)),
        "fold_aucs": [round(a, 5) for a in fold_aucs],
        "wall_s": round(elapsed, 1),
        "peak_rss_gb": round(peak_rss_gb(), 2),
    }
    results.append(row)
    oof_store[name] = oof
    test_store[name] = test_pred
    print(
        f"{name:30s} OOF AUC {row['oof_auc']:.5f} ± {row['fold_std']:.5f} "
        f"| {row['wall_s']:7.1f}s | peak RSS {row['peak_rss_gb']:.2f} GB"
    )
    print(f"{'':30s} folds: {row['fold_aucs']}")
    return oof, test_pred

## 4. v1 — Sanity Baselines (+ First-Fit Measurement)

Constant predictor (floor), regularized logistic regression (linear floor —
sanity only, not a candidate: it cannot represent the subsidy gate without
an explicit product), and default `HistGradientBoostingClassifier`. The HGB
run doubles as the measured first full-data fit.

In [4]:
if RUN_V1_SANITY:
    const_auc = roc_auc_score(y, np.full(len(y), float(y.mean())))
    print(f"v1a_constant: AUC {const_auc:.3f} (rankless floor)")

v1a_constant: AUC 0.500 (rankless floor)


In [5]:
if RUN_V1_SANITY:

    def logistic_factory():
        pre = ColumnTransformer([
            ("num", StandardScaler(), NUMERIC_FEATURES + [ANX]),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                BASE_CATEGORICALS,
            ),
        ])
        return Pipeline([
            ("pre", pre),
            ("clf", LogisticRegression(max_iter=2000)),
        ])

    run_cv("v1b_logistic", logistic_factory, X, X_test)

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


...

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


v1b_logistic                   OOF AUC 0.93809 ± 0.00081 |     5.6s | peak RSS 0.98 GB
                               folds: [np.float64(0.93667), np.float64(0.93803), np.float64(0.93907), np.float64(0.93862), np.float64(0.93807)]


/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [6]:
if RUN_V1_SANITY:
    run_cv(
        "v1c_hgb_default",
        lambda: HistGradientBoostingClassifier(
            random_state=SEED, categorical_features="from_dtype"
        ),
        X,
        X_test,
    )

v1c_hgb_default                OOF AUC 0.94102 ± 0.00087 |    18.2s | peak RSS 0.99 GB
                               folds: [np.float64(0.9395), np.float64(0.94093), np.float64(0.94193), np.float64(0.94185), np.float64(0.94099)]


**Insight:** the constant floor confirms the harness (AUC 0.500). The
logistic floor lands at **0.93809 ± 0.00081** — remarkably close to the
GBDTs (gap ≈ 0.003): despite the subsidy gate, most of the *ranking* is
recoverable additively in log-odds. Its solver emitted overflow
RuntimeWarnings (as in S6E8) — noted, not blocking: predictions are finite,
in-range, and sane. `v1c_hgb_default` reaches **0.94102 ± 0.00087** in
18.2 s at 0.99 GB peak RSS — the measured first full-data fit required by
the scale override: full-data 5-fold CV is cheap here, and no subsample
regime is needed for standard experiments.

## 5. v2 — Strong Models (Native Categorical)

Untuned LightGBM and CatBoost with native categorical handling. Untuned
deliberately — S6E8 showed the tuning comparison must be budget-matched
later, not smuggled into the baseline.

In [7]:
if RUN_V2_STRONG:
    run_cv(
        "v2a_lightgbm_default",
        lambda: lgb.LGBMClassifier(random_state=SEED, verbose=-1),
        X,
        X_test,
    )

v2a_lightgbm_default           OOF AUC 0.94115 ± 0.00082 |     8.4s | peak RSS 0.99 GB
                               folds: [np.float64(0.93998), np.float64(0.94072), np.float64(0.94235), np.float64(0.94172), np.float64(0.94107)]


In [8]:
if RUN_V2_STRONG:
    run_cv(
        "v2b_catboost_default",
        lambda: CatBoostClassifier(
            random_seed=SEED, verbose=0, allow_writing_files=False
        ),
        X,
        X_test,
        cat_features=BASE_CATEGORICALS,
    )

v2b_catboost_default           OOF AUC 0.94157 ± 0.00072 |   550.9s | peak RSS 1.14 GB
                               folds: [np.float64(0.94045), np.float64(0.94121), np.float64(0.94248), np.float64(0.94221), np.float64(0.9415)]


**Insight:** `v2a_lightgbm_default` **0.94115 ± 0.00082** (8.4 s);
`v2b_catboost_default` **0.94157 ± 0.00072** (550.9 s). All four GBDT runs
sit within 0.00055 of each other — the plateau EDA §3 predicted. CatBoost's
+0.00042 lead over LightGBM is smaller than the fold std (~0.0007–0.0009),
so this is a *working-champion* lead under the predeclared highest-OOF rule,
not a paired-gate promotion — and it costs ~65× LightGBM's runtime (CatBoost
defaults to 1000 iterations vs. LightGBM's 100, so the budgets are not
comparable; that comparison belongs to the hand-designed tuning phase).

## 6. `Range_Anxiety_Level` Representation A/B

Ordinal (default, monotone evidence) vs. native categorical, same family
and folds — the cheap A/B promised in the plan. Decided on OOF AUC delta
relative to fold std, not on a single-split number.

In [9]:
if RUN_ANX_CATEGORICAL_AB:
    run_cv(
        "v2c_lightgbm_anx_categorical",
        lambda: lgb.LGBMClassifier(random_state=SEED, verbose=-1),
        X_anxcat,
        X_test_anxcat,
    )

v2c_lightgbm_anx_categorical   OOF AUC 0.94123 ± 0.00088 |     8.7s | peak RSS 1.26 GB
                               folds: [np.float64(0.9398), np.float64(0.94097), np.float64(0.94239), np.float64(0.94184), np.float64(0.94123)]


**Insight:** categorical treatment scores 0.94123 vs. ordinal 0.94115 —
Δ = +0.00008, an order of magnitude below fold std. A tie: the ordinal
default stands (simpler, consistent with the monotone evidence), and the
A/B is logged in `docs/4_experiment_ledger.md` as resolved-no-effect, per
the plan's "logged either way" rule.

## 7. Summary, Sanity Checks, and Candidate Diversity

In [10]:
summary = (
    pd.DataFrame(results)
    .sort_values("oof_auc", ascending=False)
    .reset_index(drop=True)
)
summary

,run,oof_auc,fold_std,fold_aucs,wall_s,peak_rss_gb
0,v2b_catboost_default,0.941566,0.000723,"[0.94045, 0.94121, 0.94248, 0.94221, 0.9415]",550.9,1.14
1,v2c_lightgbm_anx_categorical,0.941230,0.000876,"[0.9398, 0.94097, 0.94239, 0.94184, 0.94123]",8.7,1.26
2,v2a_lightgbm_default,0.941150,0.000816,"[0.93998, 0.94072, 0.94235, 0.94172, 0.94107]",8.4,0.99
3,v1c_hgb_default,0.941017,0.000875,"[0.9395, 0.94093, 0.94193, 0.94185, 0.94099]",18.2,0.99
4,v1b_logistic,0.938089,0.000807,"[0.93667, 0.93803, 0.93907, 0.93862, 0.93807]",5.6,0.98


In [11]:
def candidate_sanity_checks(name: str) -> dict:
    """Finite, bounded, non-degenerate predictions; quantile comparison."""
    oof, test_pred = oof_store[name], test_store[name]
    return {
        "finite": bool(
            np.isfinite(oof).all() and np.isfinite(test_pred).all()
        ),
        "in_range": bool(
            (oof >= 0).all()
            and (oof <= 1).all()
            and (test_pred >= 0).all()
            and (test_pred <= 1).all()
        ),
        "oof_unique": int(np.unique(oof).size),
        "test_unique": int(np.unique(test_pred).size),
        "oof_q05_50_95": np.quantile(oof, [0.05, 0.5, 0.95])
        .round(4)
        .tolist(),
        "test_q05_50_95": np.quantile(test_pred, [0.05, 0.5, 0.95])
        .round(4)
        .tolist(),
    }


for name in oof_store:
    print(name, candidate_sanity_checks(name))

oof_corr = pd.DataFrame(oof_store).corr().round(4)
print("\nOOF Pearson correlation (diversity bar: blend only if <= 0.995):")
print(oof_corr.to_string())

v1b_logistic {'finite': True, 'in_range': True, 'oof_unique': 668665, 'test_unique': 286571, 'oof_q05_50_95': [0.0001, 0.024, 0.7883], 'test_q05_50_95': [0.0001, 0.0242, 0.7891]}
v1c_hgb_default {'finite': True, 'in_range': True, 'oof_unique': 662582, 'test_unique': 286449, 'oof_q05_50_95': [0.0003, 0.0189, 0.7884], 'test_q05_50_95': [0.0003, 0.0194, 0.7879]}


v2a_lightgbm_default {'finite': True, 'in_range': True, 'oof_unique': 663453, 'test_unique': 286440, 'oof_q05_50_95': [0.0002, 0.0187, 0.7903], 'test_q05_50_95': [0.0002, 0.0191, 0.7895]}


v2b_catboost_default {'finite': True, 'in_range': True, 'oof_unique': 668654, 'test_unique': 286566, 'oof_q05_50_95': [0.0001, 0.0178, 0.8028], 'test_q05_50_95': [0.0001, 0.0183, 0.7999]}
v2c_lightgbm_anx_categorical {'finite': True, 'in_range': True, 'oof_unique': 664045, 'test_unique': 286408, 'oof_q05_50_95': [0.0003, 0.0188, 0.7904], 'test_q05_50_95': [0.0003, 0.0192, 0.7897]}

OOF Pearson correlation (diversity bar: blend only if <= 0.995):
                              v1b_logistic  v1c_hgb_default  v2a_lightgbm_default  v2b_catboost_default  v2c_lightgbm_anx_categorical
v1b_logistic                        1.0000           0.9887                0.9888                0.9815                        0.9890
v1c_hgb_default                     0.9887           1.0000                0.9979                0.9937                        0.9980
v2a_lightgbm_default                0.9888           0.9979                1.0000                0.9940                        0.9991
v2b_catboost_d

**Insight:** every candidate passes sanity — finite, in `[0,1]`,
non-degenerate (>286k unique test values), and train/test prediction
quantiles align to the third decimal (consistent with the no-drift EDA
verdict). Diversity against the predeclared 0.995 bar:
CatBoost-vs-LightGBM **0.9940** and CatBoost-vs-HGB **0.9937** are *below*
the bar — a CatBoost+LightGBM blend is an eligible later experiment. The
LightGBM-family pairs (0.9979–0.9991) are not.

## 8. Working Champion & Prediction Artifacts

Working-champion rule (predeclared, `docs/4_experiment_ledger.md`): highest
OOF AUC among candidates passing sanity checks. The logistic floor is
excluded by predeclaration. Aligned OOF/test matrices are persisted locally
for later paired-gate comparisons.

In [12]:
EXCLUDED_FROM_CANDIDACY = {"v1b_logistic"}  # sanity floor only
candidates = [
    r for r in results if r["run"] not in EXCLUDED_FROM_CANDIDACY
]
champion = max(candidates, key=lambda r: r["oof_auc"])
CHAMPION_NAME = champion["run"]
print(
    f"working champion: {CHAMPION_NAME} "
    f"(OOF AUC {champion['oof_auc']:.5f})"
)

PRED_DIR = Path("../predictions")
if PRED_DIR.is_dir():  # local repo only; absent on Kaggle
    for name in oof_store:
        np.save(PRED_DIR / f"{name}_oof.npy", oof_store[name])
        np.save(PRED_DIR / f"{name}_test.npy", test_store[name])
    print(f"aligned prediction matrices saved to {PRED_DIR.resolve()}")

working champion: v2b_catboost_default (OOF AUC 0.94157)


aligned prediction matrices saved to /Users/tuannm3812/Documents/GitHub/2. Kaggle/kaggle-s6e9-predicting-electric-vehicle-purchases/predictions


## 9. Submission

In [13]:
if RUN_SUBMISSION:
    submission = pd.DataFrame(
        {"id": test["id"], TARGET: test_store[CHAMPION_NAME]}
    )
    assert submission.shape == sample_submission.shape
    assert (
        submission["id"].values == sample_submission["id"].values
    ).all()
    submission.to_csv("submission.csv", index=False)
    print(
        f"submission.csv written from {CHAMPION_NAME} "
        f"(notebook {NOTEBOOK_VERSION}); range "
        f"[{submission[TARGET].min():.4f}, "
        f"{submission[TARGET].max():.4f}]"
    )

submission.csv written from v2b_catboost_default (notebook v1); range [0.0000, 0.9950]


## 10. Next Moves

1. **Recorded:** all five runs are ledger rows in
   `docs/4_experiment_ledger.md`; working champion `v2b_catboost_default`
   (OOF AUC 0.94157, F1 folds).
2. **First submission** (hypothesis-gated): CV should track the leaderboard
   (EDA drift evidence). Push this kernel, run it on Kaggle, submit the
   completed version via `kaggle competitions submit -k
   tuannm3812/ev-purchases-baseline-modeling -v <version> -f
   submission.csv`, and log it in `docs/5_submission_manifest.md`.
3. **Budget-matched tuning next, not more families:** LightGBM's default
   100 trees vs. CatBoost's 1000 iterations means v2 compared unequal
   budgets (S6E8's E01 lesson). A small hand-designed search with
   comparable boosting budgets across HGB/LightGBM/CatBoost is the next
   experiment, with the paired promotion gate predeclared in the ledger.
4. **Blend candidate:** CatBoost + LightGBM OOF correlation 0.9940 clears
   the ≤ 0.995 diversity bar — a convex OOF-weighted blend is eligible once
   tuned single models exist, subject to the paired gate.
5. **Runtime note:** CatBoost (551 s) is the sweep bottleneck; tune
   LightGBM first, and revisit CatBoost iteration count deliberately.
6. Explicit subsidy-gate crosses stay parked unless the tuned models
   plateau below expectation.

## Reproducibility Snapshot

In [14]:
snapshot = {
    "generated_utc": datetime.now(timezone.utc).isoformat(
        timespec="seconds"
    ),
    "notebook_version": NOTEBOOK_VERSION,
    "seed": SEED,
    "fold_definition": (
        f"F1: StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, "
        f"random_state={SEED})"
    ),
    "champion": CHAMPION_NAME,
    "results": results,
    "versions": {
        m.__name__: m.__version__ for m in (np, pd, sklearn, lgb)
    },
    "python": platform.python_version(),
}
print(json.dumps(snapshot, indent=2))

{
  "generated_utc": "2026-09-01T08:40:51+00:00",
  "notebook_version": "v1",
  "seed": 42,
  "fold_definition": "F1: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
  "champion": "v2b_catboost_default",
  "results": [
    {
      "run": "v1b_logistic",
      "oof_auc": 0.9380892677192406,
      "fold_std": 0.0008073157075398691,
      "fold_aucs": [
        0.93667,
        0.93803,
        0.93907,
        0.93862,
        0.93807
      ],
      "wall_s": 5.6,
      "peak_rss_gb": 0.98
    },
    {
      "run": "v1c_hgb_default",
      "oof_auc": 0.9410172617838523,
      "fold_std": 0.000874832760267404,
      "fold_aucs": [
        0.9395,
        0.94093,
        0.94193,
        0.94185,
        0.94099
      ],
      "wall_s": 18.2,
      "peak_rss_gb": 0.99
    },
    {
      "run": "v2a_lightgbm_default",
      "oof_auc": 0.9411501450171624,
      "fold_std": 0.0008157317194749017,
      "fold_aucs": [
        0.93998,
        0.94072,
        0.94235,
        0.9